In [ ]:
%pip install pdfplumber sentence-transformers chromadb requests beautifulsoup4 numpy

In [ ]:
import pdfplumber
import re

pdf_text = ""
with pdfplumber.open("data/attention_is_all_you_need.pdf") as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            pdf_text += page_text + "\n"

pdf_text = re.sub(r'\s+', ' ', pdf_text).strip()
print(f"PDF loaded: {len(pdf_text)} characters")

PDF loaded: 35525 characters


In [5]:
import requests
from bs4 import BeautifulSoup

url = "https://en.wikipedia.org/wiki/Retrieval-augmented_generation"
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

web_text = ""
for para in soup.find_all("p"):
    text = para.get_text()
    if len(text) > 50:
        web_text += text + "\n"

print(f"Website scraped: {len(web_text)} characters")

Website scraped: 9373 characters


In [6]:
def chunk_text(text, chunk_size=500, overlap=50, source=""):
    chunks = []
    start = 0
    chunk_id = 0
    while start < len(text):
        chunk = text[start:start + chunk_size]
        if chunk.strip():
            chunks.append({
                "id": f"{source}_chunk_{chunk_id}",
                "text": chunk.strip(),
                "source": source
            })
            chunk_id += 1
        start += chunk_size - overlap
    return chunks

pdf_chunks = chunk_text(pdf_text, source="pdf")
web_chunks = chunk_text(web_text, source="web")
all_chunks = pdf_chunks + web_chunks

print(f"PDF chunks: {len(pdf_chunks)}")
print(f"Web chunks: {len(web_chunks)}")
print(f"Total chunks: {len(all_chunks)}")

PDF chunks: 79
Web chunks: 21
Total chunks: 100


In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [chunk["text"] for chunk in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Each chunk = {embeddings.shape[1]} numbers")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embeddings shape: (100, 384)
Each chunk = 384 numbers


In [8]:
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection(
    name="rag_collection",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=[chunk["id"] for chunk in all_chunks],
    embeddings=embeddings.tolist(),
    documents=[chunk["text"] for chunk in all_chunks],
    metadatas=[{"source": chunk["source"]} for chunk in all_chunks]
)

print(f"Stored {collection.count()} chunks in ChromaDB")

Stored 100 chunks in ChromaDB


In [10]:
def cosine_similarity(vec1, vec2):
    dot = np.dot(vec1, vec2)
    return dot / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

def dot_product_similarity(vec1, vec2):
    return np.dot(vec1, vec2)

def search(query, method="cosine", top_k=3):
    query_vec = model.encode([query])[0]
    scores = []
    for i, emb in enumerate(embeddings):
        if method == "cosine":
            score = cosine_similarity(query_vec, emb)
        else:
            score = dot_product_similarity(query_vec, emb)
        scores.append((i, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

def print_results(query, method="cosine", top_k=3):
    print(f"Query: {query}")
    print(f"Method: {method}")
    print("-" * 50)
    results = search(query, method, top_k)
    for rank, (idx, score) in enumerate(results, 1):
        print(f"Rank {rank} | Score: {score:.4f} | Source: {all_chunks[idx]['source']}")
        print(f"Text: {all_chunks[idx]['text'][:200]}")
        print()

In [11]:
queries = [
    "What is the attention mechanism in transformers?",
    "What is retrieval augmented generation?",
    "How do transformers work?"
]

for query in queries:
    print("=" * 60)
    print_results(query, method="cosine")

print("=" * 60)
print("COMPARISON: Cosine vs Dot Product")
print("=" * 60)
query = "What is the attention mechanism?"
cosine = search(query, "cosine")
dot = search(query, "dot")
print(f"\n{'Rank':<6}{'Cosine Score':<20}{'Dot Product Score':<20}")
print("-" * 46)
for i in range(3):
    print(f"{i+1:<6}{cosine[i][1]:<20.4f}{dot[i][1]:<20.4f}")

Query: What is the attention mechanism in transformers?
Method: cosine
--------------------------------------------------
Rank 1 | Score: 0.6693 | Source: pdf
Text: putpositions.Inthesemodels, thenumberofoperationsrequiredtorelatesignalsfromtwoarbitraryinputoroutputpositionsgrows inthedistancebetweenpositions,linearlyforConvS2SandlogarithmicallyforByteNet. Thisma

Rank 2 | Score: 0.5743 | Source: pdf
Text: attentionmechanisminsteadofsequence- alignedrecurrenceandhavebeenshowntoperformwellonsimple-languagequestionansweringand languagemodelingtasks[34]. To the best of our knowledge, however, the Transform

Rank 3 | Score: 0.5539 | Source: pdf
Text: ytirojam fo fo naciremA naciremA stnemnrevog stnemnrevog evah evah dessap dessap wen wen swal swal ecnis ecnis 9002 9002 gnikam gnikam eht eht noitartsiger noitartsiger ro ro gnitov gnitov ssecorp sse

Query: What is retrieval augmented generation?
Method: cosine
--------------------------------------------------
Rank 1 | Score: 0.6156 | Sourc